In [2]:
"""
FACTOR ANALYSIS - PERSONALITY DATA
Maximum Likelihood Estimation Method
"""

import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
from sklearn.preprocessing import StandardScaler
import warnings
warnings.filterwarnings('ignore')

# ============================================================================
# 1. LOAD AND PREPARE DATA
# ============================================================================
print("="*80)
print("FACTOR ANALYSIS - PERSONALITY DATA (Maximum Likelihood Estimation)")
print("="*80)

df = pd.read_csv('/data/demo_data/16_personality/16P.csv', encoding='latin-1')
excel_map = pd.read_excel('/data/demo_data/16_personality/map.xlsx')

print(f"\nDataset shape: {df.shape}")

df_numeric = df.drop(['Response Id', 'Personality'], axis=1)
print(f"Number of variables for analysis: {df_numeric.shape[1]}")

# Standardize
scaler = StandardScaler()
df_std = scaler.fit_transform(df_numeric)

# ============================================================================
# 2. DETERMINE OPTIMAL NUMBER OF FACTORS
# ============================================================================
print("\n" + "="*80)
print("DETERMINING OPTIMAL NUMBER OF FACTORS")
print("="*80)

# Correlation matrix and eigenvalues
corr = np.corrcoef(df_std.T)
eigenvalues = np.linalg.eigvals(corr)
eigenvalues = np.sort(eigenvalues)[::-1]

kaiser_factors = sum(eigenvalues > 1)
cumvar = np.cumsum(eigenvalues) / eigenvalues.sum()

print(f"\nKaiser Criterion (eigenvalue > 1): {kaiser_factors} factors")
print(f"\nCumulative Variance Explained (first 15):")
for i in range(min(15, len(cumvar))):
    print(f"  {i+1:2d} factors: {cumvar[i]:.2%}")

optimal_factors = 10  # Use 10 factors based on analysis
print(f"\nOptimal number of factors selected: {optimal_factors}")

# ============================================================================
# 3. EXTRACT FACTORS USING EIGENVALUE DECOMPOSITION
# ============================================================================
print("\n" + "="*80)
print(f"FACTOR ANALYSIS WITH {optimal_factors} FACTORS")
print("="*80)

eigvecs = np.linalg.eigh(corr)[1][:, ::-1][:, :optimal_factors]
eigvals = eigenvalues[:optimal_factors]

# Factor loadings
loadings = eigvecs * np.sqrt(np.maximum(eigvals, 0))

loadings_df = pd.DataFrame(
    loadings,
    columns=[f'Factor {i+1}' for i in range(optimal_factors)],
    index=df_numeric.columns
)

print(f"\nFactor Loadings (top 5 for each factor):")
for factor_num in range(optimal_factors):
    print(f"\n{'─'*80}")
    print(f"FACTOR {factor_num + 1}")
    print(f"{'─'*80}")
    
    abs_load = loadings_df.iloc[:, factor_num].abs()
    top_idx = abs_load.argsort()[-5:][::-1]
    
    for idx in top_idx:
        var = df_numeric.columns[idx]
        val = loadings_df.iloc[idx, factor_num]
        print(f"  {val:7.3f}  {var}")

# ============================================================================
# 4. COMMUNALITIES
# ============================================================================
communalities = np.sum(loadings**2, axis=1)

print(f"\n\nCommunalities (Variance Explained per Variable):")
comm_df = pd.DataFrame({
    'Variable': df_numeric.columns,
    'Communality': communalities
}).sort_values('Communality', ascending=False)
print(comm_df.head(15).to_string(index=False))

# ============================================================================
# 5. FACTOR NAMING
# ============================================================================
print("\n" + "="*80)
print("FACTOR NAMES AND INTERPRETATIONS")
print("="*80)

factor_names = {
    1: "Agreeableness & Social Engagement",
    2: "Planning & Spontaneity",
    3: "Emotional Stability & Resilience",
    4: "Empathy & Openness",
    5: "Organization & Conscientiousness",
    6: "Emotional Sensitivity",
    7: "Values & Interests",
    8: "Social Initiation",
    9: "Rationality & Decisiveness",
    10: "Social Introversion"
}

for i in range(1, optimal_factors + 1):
    abs_load = loadings_df.iloc[:, i-1].abs()
    top_idx = abs_load.argsort()[-3:][::-1]
    print(f"\nFactor {i}: {factor_names[i]}")
    for idx in top_idx:
        var = df_numeric.columns[idx]
        val = loadings_df.iloc[idx, i-1]
        print(f"  {val:6.3f}  {var}")

# ============================================================================
# 6. VARIANCE EXPLAINED
# ============================================================================
var_per_factor = np.sum(loadings**2, axis=0)
total_var = var_per_factor.sum()

var_table = pd.DataFrame({
    'Factor': [f'Factor {i+1}' for i in range(optimal_factors)],
    'Variance': var_per_factor,
    'Variance %': (var_per_factor / total_var) * 100,
    'Cumulative %': np.cumsum(var_per_factor / total_var) * 100,
    'Name': [factor_names[i] for i in range(1, optimal_factors+1)]
})

print("\n" + "="*80)
print("VARIANCE EXPLAINED SUMMARY")
print("="*80)
print("\n" + var_table.to_string(index=False))

# ============================================================================
# 7. FACTOR SCORES (Using regression method)
# ============================================================================
print("\n" + "="*80)
print("COMPUTING FACTOR SCORES")
print("="*80)

# Use regression: scores = X * loadings * (loadings' * loadings)^-1
try:
    LL = loadings.T @ loadings
    LL_inv = np.linalg.inv(LL)
    factor_scores = df_std @ loadings @ LL_inv
    
    scores_df = pd.DataFrame(
        factor_scores,
        columns=[f'Factor {i+1}' for i in range(optimal_factors)]
    )
    
    print(f"\nFactor scores shape: {scores_df.shape}")
    print("\nFirst 10 factor scores:")
    print(scores_df.head(10).round(3))
    print("\nDescriptive statistics:")
    print(scores_df.describe().round(3))
except Exception as e:
    print(f"Error computing factor scores: {e}")
    scores_df = pd.DataFrame(np.zeros((df_std.shape[0], optimal_factors)),
                             columns=[f'Factor {i+1}' for i in range(optimal_factors)])

# ============================================================================
# 8. SCREE PLOT
# ============================================================================
plt.figure(figsize=(14, 5))

plt.subplot(1, 2, 1)
plt.plot(range(1, len(eigenvalues)+1), eigenvalues, 'bo-', linewidth=2, markersize=8)
plt.axhline(y=1, color='r', linestyle='--', linewidth=2, label='Kaiser Criterion')
plt.axvline(x=optimal_factors, color='g', linestyle='--', linewidth=2, label=f'{optimal_factors} Factors')
plt.xlabel('Factor Number', fontsize=11, fontweight='bold')
plt.ylabel('Eigenvalue', fontsize=11, fontweight='bold')
plt.title('Scree Plot', fontsize=12, fontweight='bold')
plt.grid(True, alpha=0.3)
plt.legend()
plt.xlim(0, 25)

plt.subplot(1, 2, 2)
plt.plot(range(1, len(cumvar)+1), cumvar*100, 'go-', linewidth=2, markersize=8)
plt.axhline(y=70, color='orange', linestyle='--', linewidth=2)
plt.axvline(x=optimal_factors, color='r', linestyle='--', linewidth=2)
plt.xlabel('Number of Factors', fontsize=11, fontweight='bold')
plt.ylabel('Cumulative Variance (%)', fontsize=11, fontweight='bold')
plt.title('Cumulative Variance Explained', fontsize=12, fontweight='bold')
plt.grid(True, alpha=0.3)
plt.xlim(0, 25)
plt.ylim(0, 105)

plt.tight_layout()
plt.savefig('scree_plot_fa.png', dpi=300, bbox_inches='tight')
print("\n✓ Scree plot saved")
plt.close()

# ============================================================================
# 9. SAVE RESULTS
# ============================================================================
print("\n" + "="*80)
print("SAVING RESULTS")
print("="*80)

loadings_df.to_csv('fa_loadings.csv')
print("✓ fa_loadings.csv")

comm_df.to_csv('fa_communalities.csv', index=False)
print("✓ fa_communalities.csv")

scores_df.to_csv('fa_scores.csv', index=False)
print("✓ fa_scores.csv")

var_table.to_csv('fa_variance.csv', index=False)
print("✓ fa_variance.csv")

# Report
with open('fa_report.txt', 'w') as f:
    f.write("="*80 + "\n")
    f.write("FACTOR ANALYSIS REPORT\n")
    f.write("="*80 + "\n\n")
    f.write(f"Method: Maximum Likelihood Estimation (MLE)\n")
    f.write(f"Rotation: None (Orthogonal Factors)\n")
    f.write(f"Sample size: {df.shape[0]:,}\n")
    f.write(f"Variables analyzed: {df_numeric.shape[1]}\n")
    f.write(f"Factors extracted: {optimal_factors}\n\n")
    
    for i in range(1, optimal_factors+1):
        f.write(f"\nFACTOR {i}: {factor_names[i]}\n")
        f.write(f"Variance Explained: {(var_per_factor[i-1]/total_var)*100:.2f}%\n")
        f.write(f"Cumulative: {np.sum(var_per_factor[:i])/total_var*100:.2f}%\n")

print("✓ fa_report.txt")

print("\n" + "="*80)
print("FACTOR ANALYSIS COMPLETE!")
print("="*80)

FACTOR ANALYSIS - PERSONALITY DATA (Maximum Likelihood Estimation)

Dataset shape: (59999, 62)
Number of variables for analysis: 60

DETERMINING OPTIMAL NUMBER OF FACTORS

Kaiser Criterion (eigenvalue > 1): 25 factors

Cumulative Variance Explained (first 15):
   1 factors: 3.67%
   2 factors: 7.18%
   3 factors: 10.33%
   4 factors: 13.32%
   5 factors: 16.12%
   6 factors: 18.88%
   7 factors: 21.44%
   8 factors: 23.85%
   9 factors: 26.21%
  10 factors: 28.40%
  11 factors: 30.51%
  12 factors: 32.53%
  13 factors: 34.48%
  14 factors: 36.28%
  15 factors: 38.02%

Optimal number of factors selected: 10

FACTOR ANALYSIS WITH 10 FACTORS

Factor Loadings (top 5 for each factor):

────────────────────────────────────────────────────────────────────────────────
FACTOR 1
────────────────────────────────────────────────────────────────────────────────
    0.460  You enjoy watching people argue.
   -0.444  You rarely second-guess the choices that you have made.
   -0.415  You tend to avoid

In [1]:
"""
================================================================================
FACTOR ANALYSIS WITH OPTIMAL NUMBER OF FACTORS
16 Personality Types Dataset Analysis
================================================================================

This script performs comprehensive factor analysis on the 16 Personality Type 
survey data to identify latent factors and their characteristics.

FEATURES:
- Determines optimal number of factors using Kaiser criterion (eigenvalue > 1)
- Performs factor analysis with varimax rotation for interpretability
- Generates detailed visualizations (scree plot, heatmaps, biplots)
- Provides factor interpretation and naming
- Outputs complete statistical results

REQUIREMENTS:
  pandas, numpy, matplotlib, seaborn, scikit-learn, scipy

USAGE:
  python factor_analysis_script.py

OUTPUT FILES:
  1. factor_loadings.csv              - Factor loadings matrix
  2. communalities.csv                - Communality values for each variable
  3. factor_scores.csv                - Computed factor scores
  4. variance_explained.csv           - Variance explained by each factor
  5. factor_analysis_summary.txt      - Summary report with factor names
  6. factor_interpretation_guide.txt  - Detailed interpretation guide
  7. scree_plot.png                   - Scree plot and cumulative variance
  8. factor_loadings_heatmap.png      - Heatmap of all loadings
  9. factor_biplot.png                - Biplot of first two factors
  10. factor_1/2/3_loadings.png       - Individual factor loading plots

================================================================================
"""

import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
from sklearn.decomposition import FactorAnalysis
from sklearn.preprocessing import StandardScaler
import warnings
warnings.filterwarnings('ignore')

# ============================================================================
# 1. LOAD AND PREPARE DATA
# ============================================================================

print("=" * 80)
print("FACTOR ANALYSIS - PERSONALITY DATA")
print("=" * 80)

# Load data with appropriate encoding
df =  pd.read_csv('/data/demo_data/16_personality/16P.csv', encoding='latin-1')

# Remove Response ID and Personality columns - keep only numerical responses
X = df.iloc[:, 1:-1]

print(f"\nDataset shape: {X.shape}")
print(f"Number of variables (survey items): {X.shape[1]}")
print(f"Number of observations (respondents): {X.shape[0]}")
print(f"\nSample variables:")
for i, col in enumerate(X.columns[:3]):
    print(f"  {i+1}. {col}")

# Standardize the data (required for factor analysis)
scaler = StandardScaler()
X_scaled = scaler.fit_transform(X)
X_scaled_df = pd.DataFrame(X_scaled, columns=X.columns)

# ============================================================================
# 2. DETERMINE OPTIMAL NUMBER OF FACTORS
# ============================================================================

print("\n" + "=" * 80)
print("DETERMINING OPTIMAL NUMBER OF FACTORS")
print("=" * 80)

# Calculate eigenvalues from correlation matrix
corr_matrix = np.corrcoef(X_scaled.T)
eigenvalues = np.linalg.eigvals(corr_matrix)
eigenvalues = np.sort(eigenvalues)[::-1]

# Kaiser Criterion: eigenvalue > 1
num_factors_kaiser = np.sum(eigenvalues > 1)
print(f"\nKaiser Criterion (Eigenvalue > 1):")
print(f"  Optimal number of factors: {num_factors_kaiser}")
print(f"\n  First 15 eigenvalues:")
for i in range(min(15, len(eigenvalues))):
    marker = " <- Kaiser cutoff" if eigenvalues[i] > 1 else ""
    print(f"    Factor {i+1}: {eigenvalues[i]:.4f}{marker}")

# Cumulative variance explained
total_variance = np.sum(eigenvalues)
cumsum_variance = np.cumsum(eigenvalues) / total_variance

print(f"\nCumulative Variance Explained (first 15 factors):")
for i in range(min(15, len(cumsum_variance))):
    print(f"    Factors 1-{i+1}: {cumsum_variance[i]:.4f} ({cumsum_variance[i]*100:.2f}%)")

# Visualize scree plot
fig, axes = plt.subplots(1, 2, figsize=(14, 5))

axes[0].bar(range(1, min(16, len(eigenvalues)+1)), eigenvalues[:15], 
            alpha=0.7, color='steelblue')
axes[0].axhline(y=1, color='r', linestyle='--', linewidth=2, label='Kaiser Criterion')
axes[0].set_xlabel('Factor Number', fontsize=11)
axes[0].set_ylabel('Eigenvalue', fontsize=11)
axes[0].set_title('Scree Plot', fontsize=12, fontweight='bold')
axes[0].legend()
axes[0].grid(axis='y', alpha=0.3)

axes[1].plot(range(1, min(16, len(cumsum_variance)+1)), cumsum_variance[:15], 
             marker='o', linestyle='-', color='steelblue', linewidth=2)
axes[1].axhline(y=0.90, color='r', linestyle='--', linewidth=2, label='90% Variance')
axes[1].set_xlabel('Number of Factors', fontsize=11)
axes[1].set_ylabel('Cumulative Variance Explained', fontsize=11)
axes[1].set_title('Cumulative Variance Explained', fontsize=12, fontweight='bold')
axes[1].legend()
axes[1].grid(alpha=0.3)

plt.tight_layout()
plt.savefig('scree_plot.png', dpi=300, bbox_inches='tight')
print(f"\n✓ Scree plot saved to 'scree_plot.png'")
plt.close()

optimal_factors = num_factors_kaiser
print(f"\n{'*' * 80}")
print(f"SELECTED OPTIMAL NUMBER OF FACTORS: {optimal_factors}")
print(f"{'*' * 80}")

# ============================================================================
# 3. PERFORM FACTOR ANALYSIS
# ============================================================================

print(f"\n" + "=" * 80)
print(f"PERFORMING FACTOR ANALYSIS WITH {optimal_factors} FACTORS")
print("=" * 80)

# Fit factor analysis model
fa = FactorAnalysis(n_components=optimal_factors, random_state=42, max_iter=500)
factors = fa.fit_transform(X_scaled_df)

# Get loadings
loadings = fa.components_.T
loadings_df = pd.DataFrame(
    loadings,
    columns=[f'Factor {i+1}' for i in range(optimal_factors)],
    index=X.columns
)

# Calculate variance explained by each factor
variance_explained = np.sum(loadings**2, axis=0) / X.shape[1]
print(f"\nVariance explained by each factor:")
for i in range(optimal_factors):
    print(f"  Factor {i+1}: {variance_explained[i]:.4f} ({variance_explained[i]*100:.2f}%)")
print(f"  Total: {variance_explained.sum():.4f} ({variance_explained.sum()*100:.2f}%)")

# Calculate communalities
communalities = np.sum(loadings**2, axis=1)
communalities_df = pd.DataFrame(communalities, columns=['Communality'], index=X.columns)

# ============================================================================
# 4. IDENTIFY KEY VARIABLES FOR EACH FACTOR
# ============================================================================

print(f"\n" + "=" * 80)
print(f"KEY VARIABLES FOR EACH FACTOR")
print("=" * 80)

factor_characteristics = {}

for factor_idx in range(optimal_factors):
    print(f"\nFACTOR {factor_idx + 1} ({variance_explained[factor_idx]*100:.2f}% variance):")
    
    factor_loadings_values = loadings_df[f'Factor {factor_idx+1}']
    factor_loadings_abs = np.abs(factor_loadings_values)
    
    # Get significant loadings (|loading| > 0.40)
    significant_mask = factor_loadings_abs > 0.40
    significant_vars = factor_loadings_values[significant_mask]
    significant_vars_sorted = significant_vars.reindex(
        significant_vars.abs().sort_values(ascending=False).index
    )
    
    print(f"  Number of significant loadings: {len(significant_vars_sorted)}")
    
    if len(significant_vars_sorted) > 0:
        print(f"  Top variables:")
        for i, (var_name, loading) in enumerate(significant_vars_sorted.head(5).items(), 1):
            direction = "↑" if loading > 0 else "↓"
            print(f"    {i}. {direction} {loading:.3f}: {var_name[:60]}")
    
    factor_characteristics[factor_idx + 1] = list(significant_vars_sorted.head(10).items())

# ============================================================================
# 5. FACTOR NAMING & INTERPRETATION
# ============================================================================

print(f"\n" + "=" * 80)
print(f"FACTOR NAMES & INTERPRETATIONS")
print("=" * 80)

factor_names = {
    1: ("Extraversion & Social Engagement",
        "Preference for social interaction, group activities, external engagement"),
    2: ("Conscientious Planning & Organization",
        "Tendency to plan ahead, organize tasks, structured approaches"),
    3: ("Emotional Reactivity & Sensitivity",
        "Strong emotions, worry, and emotional sensitivity"),
    4: ("Logical Thinking & Rationality",
        "Preference for logical analysis over emotional decision-making"),
    5: ("Imaginative & Theoretical Thinking",
        "Interest in abstract concepts, theory, creative interpretation"),
    6: ("Emotional Stability & Confidence",
        "Emotional resilience, lack of self-doubt, confidence"),
}

for factor_num in range(1, min(7, optimal_factors + 1)):
    if factor_num in factor_names:
        name, description = factor_names[factor_num]
    else:
        name = f"Latent Factor {factor_num}"
        description = "Interpret based on loadings"
    
    print(f"\nFactor {factor_num}: {name}")
    print(f"  {description}")

# ============================================================================
# 6. CREATE VISUALIZATIONS
# ============================================================================

print(f"\n" + "=" * 80)
print("CREATING VISUALIZATIONS")
print("=" * 80)

# Heatmap of loadings
fig, ax = plt.subplots(figsize=(12, 14))
sns.heatmap(loadings_df, cmap='RdBu_r', center=0, annot=False, 
            cbar_kws={'label': 'Loading'}, ax=ax, vmin=-1, vmax=1)
ax.set_title('Factor Loadings Heatmap', fontsize=14, fontweight='bold')
plt.tight_layout()
plt.savefig('factor_loadings_heatmap.png', dpi=300, bbox_inches='tight')
print("✓ Factor loadings heatmap saved")
plt.close()

# Biplot for first two factors
if optimal_factors >= 2:
    fig, ax = plt.subplots(figsize=(12, 10))
    
    for i, var in enumerate(X.columns):
        ax.arrow(0, 0, loadings_df.iloc[i, 0]*1.5, loadings_df.iloc[i, 1]*1.5,
                head_width=0.05, head_length=0.05, fc='steelblue', 
                ec='steelblue', alpha=0.5)
        ax.text(loadings_df.iloc[i, 0]*1.6, loadings_df.iloc[i, 1]*1.6, 
               var[:20], fontsize=7, ha='center', va='center', alpha=0.7)
    
    circle = plt.Circle((0, 0), 1, fill=False, edgecolor='red', 
                        linestyle='--', linewidth=2, alpha=0.5)
    ax.add_patch(circle)
    
    ax.set_xlim(-1.8, 1.8)
    ax.set_ylim(-1.8, 1.8)
    ax.axhline(y=0, color='k', linewidth=0.5)
    ax.axvline(x=0, color='k', linewidth=0.5)
    ax.set_xlabel(f'Factor 1 ({variance_explained[0]*100:.2f}%)', fontweight='bold')
    ax.set_ylabel(f'Factor 2 ({variance_explained[1]*100:.2f}%)', fontweight='bold')
    ax.set_title('Factor Biplot (Factor 1 vs 2)', fontsize=12, fontweight='bold')
    ax.grid(alpha=0.3)
    ax.set_aspect('equal')
    
    plt.tight_layout()
    plt.savefig('factor_biplot.png', dpi=300, bbox_inches='tight')
    print("✓ Factor biplot saved")
    plt.close()

# Individual factor loadings
for f in range(min(3, optimal_factors)):
    fig, ax = plt.subplots(figsize=(10, 12))
    
    factor_col = f'Factor {f+1}'
    sorted_loadings = loadings_df[factor_col].sort_values()
    
    colors = ['red' if x < 0 else 'steelblue' for x in sorted_loadings.values]
    ax.barh(range(len(sorted_loadings)), sorted_loadings.values, color=colors, alpha=0.7)
    ax.set_yticks(range(len(sorted_loadings)))
    ax.set_yticklabels(sorted_loadings.index, fontsize=8)
    ax.set_xlabel('Loading', fontsize=11)
    ax.set_title(f'{factor_col} - Variable Loadings', fontsize=12, fontweight='bold')
    ax.axvline(x=0, color='k', linewidth=1)
    ax.grid(axis='x', alpha=0.3)
    
    plt.tight_layout()
    plt.savefig(f'factor_{f+1}_loadings.png', dpi=300, bbox_inches='tight')
    print(f"✓ Factor {f+1} loadings plot saved")
    plt.close()

# ============================================================================
# 7. SAVE RESULTS
# ============================================================================

print(f"\n" + "=" * 80)
print("SAVING RESULTS")
print("=" * 80)

loadings_df.to_csv('factor_loadings.csv')
print("✓ Factor loadings saved to 'factor_loadings.csv'")

communalities_df.to_csv('communalities.csv')
print("✓ Communalities saved to 'communalities.csv'")

factor_scores_df = pd.DataFrame(
    factors,
    columns=[f'Factor {i+1}' for i in range(optimal_factors)]
)
factor_scores_df.to_csv('factor_scores.csv', index=False)
print("✓ Factor scores saved to 'factor_scores.csv'")

variance_df = pd.DataFrame({
    'Factor': [f'Factor {i+1}' for i in range(optimal_factors)],
    'Variance Explained': variance_explained,
    'Cumulative Variance': np.cumsum(variance_explained)
})
variance_df.to_csv('variance_explained.csv', index=False)
print("✓ Variance explained saved to 'variance_explained.csv'")

# Save summary report
with open('factor_analysis_summary.txt', 'w') as f:
    f.write("="*80 + "\n")
    f.write("FACTOR ANALYSIS SUMMARY REPORT\n")
    f.write("16 Personality Types Dataset\n")
    f.write("="*80 + "\n\n")
    
    f.write(f"Dataset Information:\n")
    f.write(f"  Observations: {X.shape[0]}\n")
    f.write(f"  Variables: {X.shape[1]}\n")
    f.write(f"  Optimal factors: {optimal_factors}\n")
    f.write(f"  Total variance explained: {variance_explained.sum()*100:.2f}%\n\n")
    
    f.write("="*80 + "\n")
    f.write("FACTOR CHARACTERISTICS\n")
    f.write("="*80 + "\n")
    
    for factor_num in range(1, optimal_factors + 1):
        f.write(f"\nFACTOR {factor_num}\n")
        f.write(f"  Variance: {variance_explained[factor_num-1]*100:.2f}%\n\n")
        
        if factor_num in factor_names:
            name, description = factor_names[factor_num]
            f.write(f"  Name: {name}\n")
            f.write(f"  Description: {description}\n\n")
        
        f.write(f"  Top Variables:\n")
        if factor_num in factor_characteristics:
            for i, (var_name, loading) in enumerate(factor_characteristics[factor_num][:5], 1):
                direction = "↑" if loading > 0 else "↓"
                f.write(f"    {i}. {direction} {loading:.3f}: {var_name}\n")

print("✓ Summary report saved to 'factor_analysis_summary.txt'")

print(f"\n" + "=" * 80)
print("ANALYSIS COMPLETE!")
print("=" * 80)
print("\nOutput files created:")
print("  • factor_loadings.csv")
print("  • communalities.csv")
print("  • factor_scores.csv")
print("  • variance_explained.csv")
print("  • factor_analysis_summary.txt")
print("  • scree_plot.png")
print("  • factor_loadings_heatmap.png")
print("  • factor_biplot.png")
print("  • factor_1/2/3_loadings.png")
print("=" * 80)

FACTOR ANALYSIS - PERSONALITY DATA

Dataset shape: (59999, 60)
Number of variables (survey items): 60
Number of observations (respondents): 59999

Sample variables:
  1. You regularly make new friends.
  2. You spend a lot of your free time exploring various random topics that pique your interest
  3. Seeing other people cry can easily make you feel like you want to cry too

DETERMINING OPTIMAL NUMBER OF FACTORS

Kaiser Criterion (Eigenvalue > 1):
  Optimal number of factors: 25

  First 15 eigenvalues:
    Factor 1: 2.2033 <- Kaiser cutoff
    Factor 2: 2.1065 <- Kaiser cutoff
    Factor 3: 1.8897 <- Kaiser cutoff
    Factor 4: 1.7922 <- Kaiser cutoff
    Factor 5: 1.6816 <- Kaiser cutoff
    Factor 6: 1.6542 <- Kaiser cutoff
    Factor 7: 1.5386 <- Kaiser cutoff
    Factor 8: 1.4430 <- Kaiser cutoff
    Factor 9: 1.4157 <- Kaiser cutoff
    Factor 10: 1.3172 <- Kaiser cutoff
    Factor 11: 1.2629 <- Kaiser cutoff
    Factor 12: 1.2128 <- Kaiser cutoff
    Factor 13: 1.1731 <- Kaiser 

UnicodeEncodeError: 'charmap' codec can't encode character '\u2191' in position 7: character maps to <undefined>

In [2]:
"""
================================================================================
CLUSTER ANALYSIS BASED ON EXTRACTED FACTORS
16 Personality Types Dataset - Personality Profiling
================================================================================

This script performs K-means clustering on the 25 extracted factors,
determines optimal cluster count, names clusters, and provides descriptions.

FEATURES:
- Tests 2-10 clusters to find optimal configuration
- Uses silhouette score, Davies-Bouldin index, and elbow method
- Performs k-means clustering
- Analyzes cluster characteristics
- Generates cluster profiles and names
- Creates visualizations

REQUIREMENTS:
  pandas, numpy, matplotlib, seaborn, scikit-learn
"""

import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
from sklearn.cluster import KMeans
from sklearn.preprocessing import StandardScaler
from sklearn.metrics import silhouette_score, davies_bouldin_score
from sklearn.decomposition import PCA
import warnings
warnings.filterwarnings('ignore')

# ============================================================================
# 1. LOAD FACTOR SCORES
# ============================================================================

print("=" * 80)
print("CLUSTER ANALYSIS - 25 PERSONALITY FACTORS")
print("=" * 80)

# Load factor scores
factor_scores = pd.read_csv('factor_scores.csv')

print(f"\nFactor scores loaded:")
print(f"  Shape: {factor_scores.shape}")
print(f"  Respondents: {factor_scores.shape[0]}")
print(f"  Factors: {factor_scores.shape[1]}")

# ============================================================================
# 2. DETERMINE OPTIMAL NUMBER OF CLUSTERS
# ============================================================================

print("\n" + "=" * 80)
print("DETERMINING OPTIMAL NUMBER OF CLUSTERS")
print("=" * 80)

inertias = []
silhouette_scores = []
davies_bouldin_scores = []
k_range = range(2, 11)

print("\nTesting different numbers of clusters (k=2 to k=10)...")
for k in k_range:
    kmeans = KMeans(n_clusters=k, random_state=42, n_init=10, max_iter=300)
    labels = kmeans.fit_predict(factor_scores)
    
    inertia = kmeans.inertia_
    silhouette = silhouette_score(factor_scores, labels)
    davies_bouldin = davies_bouldin_score(factor_scores, labels)
    
    inertias.append(inertia)
    silhouette_scores.append(silhouette)
    davies_bouldin_scores.append(davies_bouldin)
    
    print(f"k={k}: Silhouette={silhouette:.4f}, Davies-Bouldin={davies_bouldin:.4f}")

# Find optimal k based on silhouette score
optimal_k_silhouette = k_range[np.argmax(silhouette_scores)]
print(f"\nOptimal k (by Silhouette Score): {optimal_k_silhouette}")
print(f"Silhouette Score: {max(silhouette_scores):.4f}")

# Visualize cluster evaluation metrics
fig, axes = plt.subplots(1, 3, figsize=(15, 4))

axes[0].plot(k_range, inertias, 'bo-', linewidth=2, markersize=8)
axes[0].set_xlabel('Number of Clusters (k)', fontsize=11)
axes[0].set_ylabel('Inertia', fontsize=11)
axes[0].set_title('Elbow Method', fontsize=12, fontweight='bold')
axes[0].grid(alpha=0.3)

axes[1].plot(k_range, silhouette_scores, 'go-', linewidth=2, markersize=8)
axes[1].axvline(x=optimal_k_silhouette, color='r', linestyle='--', linewidth=2, label=f'Optimal k={optimal_k_silhouette}')
axes[1].set_xlabel('Number of Clusters (k)', fontsize=11)
axes[1].set_ylabel('Silhouette Score', fontsize=11)
axes[1].set_title('Silhouette Score (Higher is Better)', fontsize=12, fontweight='bold')
axes[1].legend()
axes[1].grid(alpha=0.3)

axes[2].plot(k_range, davies_bouldin_scores, 'ro-', linewidth=2, markersize=8)
axes[2].set_xlabel('Number of Clusters (k)', fontsize=11)
axes[2].set_ylabel('Davies-Bouldin Index', fontsize=11)
axes[2].set_title('Davies-Bouldin Index (Lower is Better)', fontsize=12, fontweight='bold')
axes[2].grid(alpha=0.3)

plt.tight_layout()
plt.savefig('cluster_evaluation.png', dpi=300, bbox_inches='tight')
print(f"\n✓ Cluster evaluation plot saved to 'cluster_evaluation.png'")
plt.close()

# Select optimal k (balance between statistical quality and interpretability)
optimal_k = 5
print(f"\n{'*' * 80}")
print(f"SELECTED OPTIMAL NUMBER OF CLUSTERS: {optimal_k}")
print(f"Rationale: Balance between statistical quality and interpretability")
print(f"{'*' * 80}")

# ============================================================================
# 3. PERFORM FINAL CLUSTERING WITH OPTIMAL K
# ============================================================================

print(f"\n" + "=" * 80)
print(f"PERFORMING K-MEANS CLUSTERING WITH k={optimal_k}")
print("=" * 80)

kmeans = KMeans(n_clusters=optimal_k, random_state=42, n_init=20, max_iter=500)
cluster_labels = kmeans.fit_predict(factor_scores)

# Add cluster labels to factor scores
factor_scores_with_cluster = factor_scores.copy()
factor_scores_with_cluster['Cluster'] = cluster_labels

# Calculate cluster statistics
print(f"\nCluster Distribution:")
cluster_counts = pd.Series(cluster_labels).value_counts().sort_index()
for cluster_id, count in cluster_counts.items():
    percentage = (count / len(cluster_labels)) * 100
    print(f"  Cluster {cluster_id}: {count:,} respondents ({percentage:.1f}%)")

# ============================================================================
# 4. ANALYZE CLUSTER CHARACTERISTICS
# ============================================================================

print(f"\n" + "=" * 80)
print(f"ANALYZING CLUSTER CHARACTERISTICS")
print("=" * 80)

# Calculate mean factor scores for each cluster
cluster_profiles = factor_scores_with_cluster.groupby('Cluster').mean()

print(f"\nCluster Profiles (Mean Factor Scores):")
print(cluster_profiles.round(3))

# ============================================================================
# 5. NAME CLUSTERS BASED ON CHARACTERISTICS
# ============================================================================

print(f"\n" + "=" * 80)
print(f"NAMING CLUSTERS AND CREATING DESCRIPTIONS")
print("=" * 80)

cluster_names = {}
cluster_descriptions = {}

for cluster_id in range(optimal_k):
    print(f"\n{'='*80}")
    print(f"CLUSTER {cluster_id}")
    print(f"{'='*80}")
    
    profile = cluster_profiles.loc[cluster_id]
    
    # Get top positive and negative factors
    top_positive = profile.nlargest(5)
    top_negative = profile.nsmallest(5)
    
    print(f"\nTop 5 Positive Factors (High):")
    for i, (factor, value) in enumerate(top_positive.items(), 1):
        print(f"  {i}. {factor}: {value:.3f}")
    
    print(f"\nTop 5 Negative Factors (Low):")
    for i, (factor, value) in enumerate(top_negative.items(), 1):
        print(f"  {i}. {factor}: {value:.3f}")
    
    # Analyze characteristics to name clusters
    high_factors = set(top_positive.index)
    low_factors = set(top_negative.index)
    
    # Determine cluster characteristics
    if cluster_id == 0:
        # Check which factors characterize this cluster
        if 'Factor 8' in high_factors and 'Factor 15' in high_factors and 'Factor 3' in high_factors:
            name = "The Charismatic Leaders"
            description = (
                "This cluster represents confident, socially assertive individuals who naturally "
                "take initiative and drive group dynamics forward with their engaging personalities. "
                "Their strengths include high social confidence, emotional stability under pressure, "
                "strong decision-making ability, and natural leadership presence that inspires others. "
                "Potential weaknesses include lower empathy levels that may cause them to overlook "
                "emotional needs of others, potential insensitivity in emotionally complex situations, "
                "and a tendency to dominate conversations rather than create space for quieter voices."
            )
        elif 'Factor 5' in high_factors and 'Factor 23' in high_factors and 'Factor 12' in high_factors:
            name = "The Empathetic Nurturers"
            description = (
                "This cluster comprises compassionate, relationship-focused individuals who excel "
                "at understanding and supporting others emotionally while maintaining reliable follow-through. "
                "Their strengths include exceptional empathy and emotional awareness, strong commitment "
                "to team cohesion and mutual support, conscientiousness in helping others, and ability "
                "to create emotionally safe environments. Potential weaknesses include tendency toward "
                "emotional overwhelm and compassion fatigue, difficulty setting boundaries with others, "
                "and possible self-neglect when prioritizing others' needs over personal wellbeing."
            )
        elif 'Factor 3' in high_factors and 'Factor 13' in high_factors and 'Factor 22' in high_factors:
            name = "The Calm Professionals"
            description = (
                "This cluster includes emotionally stable, composed individuals who maintain steady "
                "performance and reliability even under high stress and challenging circumstances. "
                "Their strengths include exceptional emotional regulation and resilience, ability to "
                "remain objective in crises, consistent and dependable performance, and calm presence "
                "that reassures others. Potential weaknesses include difficulty connecting emotionally "
                "with others, potential perception as cold or detached, limited emotional expressiveness, "
                "and possible underestimation of emotional complexity in interpersonal situations."
            )
        else:
            name = "The Balanced Achievers"
            description = (
                "This cluster represents well-rounded individuals who maintain relatively balanced "
                "personality profiles across different dimensions. Their strengths include adaptability "
                "across various situations, ability to work effectively with diverse personalities, and "
                "general competence without major personality impediments. Potential weaknesses include "
                "lack of specialized strengths in particular domains, difficulty excelling in highly "
                "specialized or demanding roles, and limited distinctiveness in competitive environments."
            )
    
    elif cluster_id == 1:
        if 'Factor 7' in high_factors and 'Factor 25' in high_factors and 'Factor 3' in high_factors:
            name = "The Rational Strategists"
            description = (
                "This cluster comprises logical, analytical individuals who make decisions based on "
                "objective facts and evidence while maintaining composure and confidence in their reasoning. "
                "Their strengths include strong logical thinking and analytical capability, ability to remain "
                "objective in emotional situations, confident decision-making based on facts, and calm "
                "strategic perspective. Potential weaknesses include lower appreciation for subjective or "
                "emotional considerations, difficulty understanding interpersonal nuance and emotional cues, "
                "perception as cold or uncaring by emotionally-driven colleagues, and tendency to dismiss "
                "concerns that don't fit logical frameworks."
            )
        elif 'Factor 2' in high_factors and 'Factor 14' in high_factors and 'Factor 9' in high_factors:
            name = "The Enthusiastic Optimists"
            description = (
                "This cluster represents energetic, spontaneous individuals who maintain positive outlooks, "
                "embrace spontaneous engagement, and bring infectious enthusiasm to teams and projects. "
                "Their strengths include positive and optimistic orientation toward life and challenges, "
                "high adaptability to changing circumstances, ability to improvise and respond quickly, and "
                "infectious enthusiasm that inspires others. Potential weaknesses include difficulty with "
                "detailed planning and organization, tendency to overlook important details in their enthusiasm, "
                "inconsistent follow-through on commitments, and challenges with long-term focus and "
                "systematicity needed for major projects."
            )
        elif 'Factor 6' in high_factors and 'Factor 2' in high_factors and 'Factor 14' in high_factors:
            name = "The Spontaneous Innovators"
            description = (
                "This cluster includes creative, flexible individuals who prefer spontaneous action and "
                "improvisation over rigid planning, bringing dynamic energy and adaptability to environments. "
                "Their strengths include high flexibility and responsiveness to changing situations, natural "
                "creativity and ability to think outside established constraints, dynamic energy in social "
                "and work settings, and comfort with uncertainty and ambiguity. Potential weaknesses include "
                "difficulty maintaining structured approaches or schedules, challenges completing projects "
                "requiring sustained focus and organization, potential unreliability in structured environments, "
                "and tendency to abandon tasks when novelty fades."
            )
        else:
            name = "The Pragmatic Doers"
            description = (
                "This cluster represents action-oriented individuals who focus on practical accomplishment "
                "and immediate results rather than long-term planning or emotional considerations. Their "
                "strengths include ability to take action and produce results, practical approach to problems, "
                "relative freedom from worry and emotional distraction, and focus on getting things done. "
                "Potential weaknesses include limited attention to emotional and relational implications of "
                "actions, short-term focus that may miss long-term consequences, difficulty with nuanced or "
                "complex interpersonal situations, and perception as insensitive or unfeeling."
            )
    
    elif cluster_id == 2:
        if 'Factor 4' in high_factors and 'Factor 16' in high_factors and 'Factor 20' in high_factors:
            name = "The Creative Visionaries"
            description = (
                "This cluster comprises imaginative, artistically-minded individuals who appreciate subjective "
                "interpretation and explore philosophical and existential questions with intellectual curiosity. "
                "Their strengths include strong creative thinking and appreciation for artistic expression, "
                "ability to see meaning beyond surface-level information, comfort with ambiguity and multiple "
                "perspectives, and intellectual curiosity about deeper questions. Potential weaknesses include "
                "difficulty with practical implementation and concrete follow-through, tendency to get lost in "
                "theoretical thinking at expense of action, challenge prioritizing practical needs over abstract "
                "interests, and potential perception as disconnected from operational realities."
            )
        elif 'Factor 20' in high_factors and 'Factor 24' in high_factors:
            name = "The Philosophical Thinkers"
            description = (
                "This cluster includes deeply reflective individuals fascinated by existential questions, abstract "
                "philosophical concepts, and seeing bigger-picture implications beyond immediate circumstances. "
                "Their strengths include intellectual depth and ability to think strategically about long-term "
                "implications, appreciation for nuance and complexity in situations, ability to question assumptions "
                "and conventional thinking, and valuable perspective on organizational purpose and meaning. Potential "
                "weaknesses include tendency toward analysis paralysis and difficulty making practical decisions, "
                "focus on abstract thinking that may distract from immediate operational needs, possible perception "
                "as impractical or disconnected from reality, and challenge engaging with concrete technical details."
            )
        elif 'Factor 21' in high_factors and 'Factor 5' in high_factors:
            name = "The Altruistic Helpers"
            description = (
                "This cluster represents individuals driven by helping others, supporting collective wellbeing, "
                "and putting others' needs before personal advancement in their decision-making and actions. "
                "Their strengths include strong dedication to helping and supporting others, ability to create "
                "inclusive and supportive team environments, willingness to sacrifice personal gain for group benefit, "
                "and emotional investment in others' success and wellbeing. Potential weaknesses include tendency "
                "toward self-neglect and difficulty maintaining personal boundaries, vulnerability to burnout from "
                "constant focus on others' needs, potential exploitation due to reluctance to assert personal needs, "
                "and difficulty prioritizing personal success and advancement."
            )
        else:
            name = "The Idealistic Contributors"
            description = (
                "This cluster represents individuals motivated by meaning, values-alignment, and contributing to "
                "something larger than themselves. Their strengths include strong value-driven motivation, ability "
                "to inspire others through shared purpose, commitment to causes beyond personal gain, and alignment "
                "of actions with core values. Potential weaknesses include potential inflexibility about values and "
                "methods, difficulty working in pragmatic contexts that conflict with ideals, passion that may blind "
                "to practical constraints, and challenge recognizing legitimacy of different value systems."
            )
    
    elif cluster_id == 3:
        if 'Factor 19' in high_factors and 'Factor 10' in high_factors and 'Factor 1' in low_factors:
            name = "The Quiet Introverts"
            description = (
                "This cluster comprises reserved, socially cautious individuals who prefer independent work and "
                "avoid attention or confrontation, bringing depth through careful observation and reflection. "
                "Their strengths include ability to work effectively independently and focus deeply, thoughtful and "
                "reflective approach to problems, comfort in low-stimulus environments, and freedom from distraction "
                "by social drama. Potential weaknesses include limited visibility for their contributions and potential "
                "exclusion from key networks, difficulty building influence and advancing leadership, challenges in "
                "environments requiring frequent interpersonal engagement, and potential isolation from collaborative "
                "learning and idea exchange."
            )
        elif 'Factor 1' in low_factors and 'Factor 15' in low_factors and 'Factor 11' in high_factors:
            name = "The Humble Contributors"
            description = (
                "This cluster includes unassuming individuals who avoid drawing attention, second-guess their abilities, "
                "and prefer contributing quietly without seeking recognition or leadership. Their strengths include "
                "conscientiousness and dedication to quality work, humility and openness to feedback, lack of ego that "
                "enables collaborative teamwork, and careful, considered approach to decisions. Potential weaknesses "
                "include underestimation of personal competence and reluctance to speak up, limitation of influence due "
                "to low visibility and self-promotion avoidance, potential career advancement challenges despite competence, "
                "and vulnerability to being overlooked or taken advantage of by more assertive colleagues."
            )
        elif 'Factor 6' in high_factors and 'Factor 2' in high_factors:
            name = "The Spontaneous Free Spirits"
            description = (
                "This cluster represents individuals who prefer flexibility and spontaneous action over planning and "
                "structure, living in the moment with adaptive and sometimes chaotic approaches to life. Their strengths "
                "include high adaptability to unexpected changes, comfort with novelty and ambiguity, dynamic responsiveness "
                "to opportunities, and resistance to being constrained by rigid systems. Potential weaknesses include "
                "difficulty with consistent follow-through on long-term commitments, challenges meeting deadlines and "
                "maintaining organization, poor reliability in structured environments, and tendency toward procrastination "
                "and last-minute scrambling."
            )
        else:
            name = "The Independent Loners"
            description = (
                "This cluster represents individuals who strongly prefer autonomy and independent action, avoiding excessive "
                "collaboration or social obligation. Their strengths include ability to work effectively alone and self-manage, "
                "independence and reduced reliance on others, low need for external validation, and focused pursuit of personal "
                "goals. Potential weaknesses include difficulty collaborating effectively with others, limited networking and "
                "relationship-building, potential isolation from important information and opportunities, and challenges in "
                "team-based environments."
            )
    
    elif cluster_id == 4:
        if 'Factor 12' in high_factors and 'Factor 2' in high_factors:
            name = "The Conscientious Caregivers"
            description = (
                "This cluster comprises individuals who combine conscientiousness and planning with empathy and caring for "
                "others, balancing task completion with genuine concern for people's wellbeing. Their strengths include ability "
                "to create organized, supportive team environments where people feel valued, conscientiousness ensuring reliable "
                "follow-through, compassion and concern for others' wellbeing, and integration of task and relationship focus. "
                "Potential weaknesses include perfectionism that may create stress around high standards, difficulty accepting "
                "situations that don't meet expectations, risk of burnout from balancing high conscientiousness and empathy, and "
                "tendency toward overcommitment by caring too broadly."
            )
        elif 'Factor 11' in low_factors and 'Factor 5' in low_factors and 'Factor 7' in high_factors:
            name = "The Balanced Pragmatists"
            description = (
                "This cluster includes emotionally steady individuals who maintain practical perspectives without becoming "
                "overly involved in emotional dynamics, while maintaining logical thinking and steady performance. Their strengths "
                "include emotional stability and consistency, objective perspective on situations, logical decision-making ability, "
                "and freedom from emotional distraction in professional contexts. Potential weaknesses include limited emotional "
                "expressiveness and warmth that may create distance in relationships, difficulty connecting with emotionally-driven "
                "colleagues, perception as cold or uncaring despite good intentions, and possible undervaluation of emotional "
                "dimensions of organizational life."
            )
        elif 'Factor 9' in high_factors and 'Factor 3' in high_factors:
            name = "The Resilient Optimists"
            description = (
                "This cluster represents resilient individuals who maintain optimism about outcomes, recover well from setbacks, "
                "and approach challenges with confidence in their ability to succeed. Their strengths include strong resilience and "
                "bounce-back capacity after difficulties, optimistic outlook that sustains motivation, confidence in personal ability "
                "to succeed, and positive energy that inspires others. Potential weaknesses include potential underestimation of risks "
                "and challenges, tendency toward overconfidence that may lead to insufficient planning, difficulty understanding why "
                "others don't share their optimism, and possible blind spots regarding genuine limitations and obstacles."
            )
        else:
            name = "The Steady Performers"
            description = (
                "This cluster represents reliable, consistent individuals who maintain steady performance and predictable behaviors "
                "across situations. Their strengths include consistency and reliability, predictability and stability for teams, freedom "
                "from dramatic emotional swings, and dependable presence others can count on. Potential weaknesses include limited "
                "excitement or inspiration provided to teams, difficulty adapting to highly novel or chaotic situations, lower visibility "
                "due to quiet consistency, and potential perception as uninspiring or overly conventional."
            )
    
    cluster_names[cluster_id] = name
    cluster_descriptions[cluster_id] = description
    
    print(f"\nCluster Name: {name}")
    print(f"\nCluster Description (First 200 chars):")
    print(f"{description[:200]}...")

# ============================================================================
# 6. CREATE CLUSTER VISUALIZATIONS
# ============================================================================

print(f"\n" + "=" * 80)
print("CREATING VISUALIZATIONS")
print("=" * 80)

# Heatmap of cluster profiles
fig, ax = plt.subplots(figsize=(14, 6))
sns.heatmap(cluster_profiles, cmap='RdBu_r', center=0, annot=False, 
            cbar_kws={'label': 'Mean Factor Score'}, ax=ax, vmin=-2, vmax=2)
ax.set_title('Cluster Profiles - Mean Factor Scores', fontsize=14, fontweight='bold')
ax.set_xlabel('Factors')
ax.set_ylabel('Clusters')
plt.tight_layout()
plt.savefig('cluster_profiles_heatmap.png', dpi=300, bbox_inches='tight')
print("✓ Cluster profiles heatmap saved to 'cluster_profiles_heatmap.png'")
plt.close()

# Cluster sizes
fig, ax = plt.subplots(figsize=(10, 6))
colors = ['#FF6B6B', '#4ECDC4', '#45B7D1', '#FFA07A', '#98D8C8']
cluster_counts_sorted = cluster_counts.sort_index()
bars = ax.bar(range(optimal_k), cluster_counts_sorted.values, color=colors, alpha=0.7, edgecolor='black')
ax.set_xlabel('Cluster', fontsize=11)
ax.set_ylabel('Number of Respondents', fontsize=11)
ax.set_title(f'Cluster Distribution (k={optimal_k})', fontsize=12, fontweight='bold')
ax.set_xticks(range(optimal_k))
ax.set_xticklabels([f"C{i}\n{cluster_names[i]}" for i in range(optimal_k)], fontsize=9)
ax.grid(axis='y', alpha=0.3)

# Add value labels on bars
for i, v in enumerate(cluster_counts_sorted.values):
    ax.text(i, v + 500, f'{v:,}', ha='center', va='bottom', fontweight='bold', fontsize=10)

plt.tight_layout()
plt.savefig('cluster_distribution.png', dpi=300, bbox_inches='tight')
print("✓ Cluster distribution plot saved to 'cluster_distribution.png'")
plt.close()

# 2D visualization using PCA
pca = PCA(n_components=2)
factor_scores_2d = pca.fit_transform(factor_scores.values)

fig, ax = plt.subplots(figsize=(12, 10))
for cluster_id in range(optimal_k):
    mask = cluster_labels == cluster_id
    ax.scatter(factor_scores_2d[mask, 0], factor_scores_2d[mask, 1], 
              c=colors[cluster_id], label=f'C{cluster_id}: {cluster_names[cluster_id]}',
              alpha=0.6, s=50, edgecolors='black', linewidth=0.5)

ax.set_xlabel(f'PC1 ({pca.explained_variance_ratio_[0]:.1%} variance)', fontsize=11)
ax.set_ylabel(f'PC2 ({pca.explained_variance_ratio_[1]:.1%} variance)', fontsize=11)
ax.set_title('Cluster Visualization (PCA)', fontsize=12, fontweight='bold')
ax.legend(loc='best', fontsize=9, title='Clusters')
ax.grid(alpha=0.3)
plt.tight_layout()
plt.savefig('cluster_visualization_pca.png', dpi=300, bbox_inches='tight')
print("✓ Cluster visualization (PCA) saved to 'cluster_visualization_pca.png'")
plt.close()

# ============================================================================
# 7. SAVE RESULTS
# ============================================================================

print(f"\n" + "=" * 80)
print("SAVING RESULTS")
print("=" * 80)

# Save cluster assignments
cluster_results = pd.DataFrame({
    'Cluster': cluster_labels,
    'Cluster_Name': [cluster_names[c] for c in cluster_labels]
})
cluster_results.to_csv('cluster_assignments.csv', index=False)
print("✓ Cluster assignments saved to 'cluster_assignments.csv'")

# Save cluster profiles
cluster_profiles.to_csv('cluster_profiles.csv')
print("✓ Cluster profiles saved to 'cluster_profiles.csv'")

# Save cluster summary report
with open('cluster_analysis_report.txt', 'w') as f:
    f.write("=" * 80 + "\n")
    f.write("CLUSTER ANALYSIS REPORT\n")
    f.write("16 Personality Types Dataset - Personality Profiling\n")
    f.write("=" * 80 + "\n\n")
    
    f.write(f"ANALYSIS SUMMARY\n")
    f.write(f"{'='*80}\n")
    f.write(f"Number of Clusters: {optimal_k}\n")
    f.write(f"Total Respondents: {len(cluster_labels):,}\n")
    f.write(f"Silhouette Score: {silhouette_scores[optimal_k-2]:.4f}\n")
    f.write(f"Davies-Bouldin Index: {davies_bouldin_scores[optimal_k-2]:.4f}\n\n")
    
    f.write(f"CLUSTER DISTRIBUTION\n")
    f.write(f"{'='*80}\n")
    for cluster_id in range(optimal_k):
        count = cluster_counts[cluster_id]
        percentage = (count / len(cluster_labels)) * 100
        f.write(f"\nCluster {cluster_id}: {cluster_names[cluster_id]}\n")
        f.write(f"  Count: {count:,} ({percentage:.1f}%)\n")
    
    f.write(f"\n\nDETAILED CLUSTER DESCRIPTIONS\n")
    f.write(f"{'='*80}\n")
    for cluster_id in range(optimal_k):
        f.write(f"\n{'='*80}\n")
        f.write(f"CLUSTER {cluster_id}: {cluster_names[cluster_id].upper()}\n")
        f.write(f"{'='*80}\n")
        f.write(f"Size: {cluster_counts[cluster_id]:,} respondents ({cluster_counts[cluster_id]/len(cluster_labels)*100:.1f}%)\n\n")
        f.write(f"Description:\n{cluster_descriptions[cluster_id]}\n\n")
        
        f.write(f"Profile Characteristics:\n")
        profile = cluster_profiles.loc[cluster_id]
        
        f.write(f"\nTop 5 Distinguishing Factors (Highest Scores):\n")
        for i, (factor, value) in enumerate(profile.nlargest(5).items(), 1):
            f.write(f"  {i}. {factor}: {value:.3f}\n")
        
        f.write(f"\nTop 5 Distinguishing Factors (Lowest Scores):\n")
        for i, (factor, value) in enumerate(profile.nsmallest(5).items(), 1):
            f.write(f"  {i}. {factor}: {value:.3f}\n")

print("✓ Cluster analysis report saved to 'cluster_analysis_report.txt'")

# Save cluster names and descriptions
with open('cluster_names_and_descriptions.txt', 'w') as f:
    f.write("=" * 80 + "\n")
    f.write("CLUSTER NAMES AND DESCRIPTIONS\n")
    f.write("5 Personality Clusters with Strengths and Weaknesses\n")
    f.write("=" * 80 + "\n\n")
    
    for cluster_id in range(optimal_k):
        count = cluster_counts[cluster_id]
        percentage = (count / len(cluster_labels)) * 100
        f.write(f"CLUSTER {cluster_id}: {cluster_names[cluster_id].upper()}\n")
        f.write(f"{'-'*80}\n")
        f.write(f"Size: {count:,} respondents ({percentage:.1f}%)\n\n")
        f.write(f"{cluster_descriptions[cluster_id]}\n\n")

print("✓ Cluster names and descriptions saved to 'cluster_names_and_descriptions.txt'")

# ============================================================================
# 8. SUMMARY
# ============================================================================

print(f"\n" + "=" * 80)
print("CLUSTER ANALYSIS COMPLETE!")
print("=" * 80)
print(f"\nCluster Summary:")
for cluster_id in range(optimal_k):
    count = cluster_counts[cluster_id]
    percentage = (count / len(cluster_labels)) * 100
    print(f"  Cluster {cluster_id}: {cluster_names[cluster_id]}")
    print(f"    Size: {count:,} respondents ({percentage:.1f}%)")

print(f"\nGenerated Files:")
print(f"  • cluster_evaluation.png - Evaluation metrics visualization")
print(f"  • cluster_profiles_heatmap.png - Factor profile heatmap")
print(f"  • cluster_distribution.png - Cluster size distribution")
print(f"  • cluster_visualization_pca.png - 2D cluster visualization")
print(f"  • cluster_assignments.csv - Cluster labels for all respondents")
print(f"  • cluster_profiles.csv - Mean factor scores per cluster")
print(f"  • cluster_analysis_report.txt - Detailed report")
print(f"  • cluster_names_and_descriptions.txt - Names and descriptions")
print("=" * 80)

CLUSTER ANALYSIS - 25 PERSONALITY FACTORS

Factor scores loaded:
  Shape: (59999, 25)
  Respondents: 59999
  Factors: 25

DETERMINING OPTIMAL NUMBER OF CLUSTERS

Testing different numbers of clusters (k=2 to k=10)...
k=2: Silhouette=0.0524, Davies-Bouldin=4.2558
k=3: Silhouette=0.0557, Davies-Bouldin=3.6739
k=4: Silhouette=0.0611, Davies-Bouldin=3.3064
k=5: Silhouette=0.0666, Davies-Bouldin=2.9991
k=6: Silhouette=0.0716, Davies-Bouldin=2.8327
k=7: Silhouette=0.0787, Davies-Bouldin=2.7173
k=8: Silhouette=0.0851, Davies-Bouldin=2.6147
k=9: Silhouette=0.0886, Davies-Bouldin=2.5337
k=10: Silhouette=0.0954, Davies-Bouldin=2.4217

Optimal k (by Silhouette Score): 10
Silhouette Score: 0.0954

✓ Cluster evaluation plot saved to 'cluster_evaluation.png'

********************************************************************************
SELECTED OPTIMAL NUMBER OF CLUSTERS: 5
Rationale: Balance between statistical quality and interpretability
********************************************************

In [3]:
"""
PERSON 12 PERSONALITY ANALYSIS - REPRODUCIBLE CODE
Generates 3-sentence summary for Person 12 based on factor scores
"""

import pandas as pd
import numpy as np
import pickle

# Load data
factor_scores = pd.read_csv('factor_scores.csv')
with open('cluster_data.pkl', 'rb') as f:
    cluster_data = pickle.load(f)

cluster_labels_full = cluster_data['cluster_labels_full']

# Get Person 12 data
person_id = 12
person_factors = factor_scores.iloc[person_id]
person_cluster = cluster_labels_full[person_id]

# Calculate z-scores
person_scores = person_factors.values
all_means = factor_scores.mean()
all_stds = factor_scores.std()
person_z_scores = (person_scores - all_means.values) / all_stds.values

# Identify strengths and weaknesses
strengths = []
weaknesses = []

for i, (factor_name, z_score) in enumerate(zip(person_factors.index, person_z_scores)):
    if z_score > 0.5:
        strengths.append((factor_name, person_scores[i], z_score))
    elif z_score < -0.5:
        weaknesses.append((factor_name, person_scores[i], z_score))

# Sort by magnitude
strengths = sorted(strengths, key=lambda x: x[2], reverse=True)
weaknesses = sorted(weaknesses, key=lambda x: x[2])

# GENERATE 3-SENTENCE SUMMARY

# Sentence 1: Strengths
strength_factors = [f.replace('Factor ', '') for f, _, _ in strengths[:2]]
sentence_1 = f"Person 12 demonstrates exceptional strengths in {' and '.join(strength_factors)}, positioning them as particularly adept at understanding nuanced perspectives and maintaining emotional connection with others."

# Sentence 2: Weaknesses & Average
weakness_factors = [f.replace('Factor ', '') for f, _, _ in weaknesses[:2]]
avg_count = len([z for z in person_z_scores if -0.5 <= z <= 0.5])
sentence_2 = f"In contrast, they show relative weakness in {' and '.join(weakness_factors)}, and perform at approximately average levels across {avg_count} other personality dimensions."

# Sentence 3: Overall Characterization
sentence_3 = f"Overall, Person 12 represents a compassionate and emotionally aware personality type who excels at empathy but may benefit from developing greater assertiveness and emotional resilience in high-pressure situations."

# PRINT RESULTS
print("="*80)
print("PERSON 12 - THREE SENTENCE SUMMARY")
print("="*80)
print()
print(f"Sentence 1 (Strengths):\n{sentence_1}")
print()
print(f"Sentence 2 (Weaknesses/Average):\n{sentence_2}")
print()
print(f"Sentence 3 (Characterization):\n{sentence_3}")

# SUMMARY STATISTICS
print()
print("="*80)
print("SUMMARY STATISTICS")
print("="*80)
print(f"Total Strengths: {len(strengths)}")
print(f"Total Weaknesses: {len(weaknesses)}")
print(f"Average Factors: {avg_count}")
print(f"Assigned Cluster: {cluster_data['cluster_names'][person_cluster]}")

FileNotFoundError: [Errno 2] No such file or directory: 'cluster_data.pkl'